In [5]:
import pandas as pd
import numpy as np

# LOAD DATASET
df = pd.read_csv("verbo_data.csv")

df.columns = df.columns.str.strip()
columns_to_keep = [
    "price_per_night",
    "listing_city",
    "checkin_date",
    "checkout_date",
    "nights",
    'total_price'
]

df = df[columns_to_keep].copy()

# NUMERIC CLEANING
df["price_per_night"] = pd.to_numeric(df["price_per_night"], errors="coerce")
df["total_price"] = pd.to_numeric(df["total_price"], errors="coerce")
df["nights"] = pd.to_numeric(df["nights"], errors="coerce")

# MISSING VALUES
df["price_per_night"] = df["price_per_night"].fillna(df["price_per_night"].median())
df["total_price"] = df["total_price"].fillna(df["total_price"].median())
df["nights"] = df["nights"].fillna(df["nights"].median())

df = df.drop_duplicates()

df["checkin_date"] = pd.to_datetime(df["checkin_date"], errors="coerce")
df["checkout_date"] = pd.to_datetime(df["checkout_date"], errors="coerce")

# FEATURE ENGINEERING
df["nights_calculated"] = (df["checkout_date"] - df["checkin_date"]).dt.days
df["nights"] = df["nights"].fillna(df["nights_calculated"])
mean_nights = int(round(df["nights"].dropna().mean()))
df.loc[(df["nights"] <= 0) | (df["nights"].isna()), "nights"] = mean_nights

# Create weekday feature
df["checkin_weekday"] = df["checkin_date"].dt.dayofweek.fillna(1).astype(int)

# CITY CLEANING
city_map = {
    "la": "los_angeles",
    "los angeles": "los_angeles",
    "nyc": "new_york",
    "new york": "new_york",
    "sf": "san_francisco",
    "san fran": "san_francisco",
    "san francisco": "san_francisco"
}

df["listing_city"] = (
    df["listing_city"]
    .astype(str)
    .str.lower()
    .str.strip()
    .replace(city_map)
)

df["listing_city"] = df["listing_city"].replace("nan", "unknown")
df = df[df["listing_city"] != "unknown"]

# Create weekday feature as names instead of numbers
df["checkin_weekday"] = (
    df["checkin_date"]
    .dt.day_name()
    .fillna("Monday")
)

df["price_per_night"] = (df["price_per_night"] - df["price_per_night"].mean()) / df["price_per_night"].std()
df["total_price"] = (df["total_price"] - df["total_price"].mean()) / df["total_price"].std()
nights_mean = df["nights"].mean()
nights_std = df["nights"].std()
df["nights"] = (df["nights"] - nights_mean) / nights_std

df = df.drop(columns=["nights_calculated", "checkin_date", "checkout_date"])

df = pd.get_dummies(df, columns=["listing_city"], prefix="", prefix_sep="", dtype=int)
df = pd.get_dummies(
    df,
    columns=["checkin_weekday"],
    prefix="",
    prefix_sep="",
    dtype=int
)

city_columns = ["los_angeles", "new_york", "san_francisco"]
city_columns = [col for col in city_columns if col in df.columns]
weekday_columns = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday_columns = [col for col in weekday_columns if col in df.columns]
price_per_night_column = ["price_per_night"]
total_price_column = ["total_price"]
feature_columns = city_columns + weekday_columns + price_per_night_column + total_price_column

# scale columns
print("🎯 FEATURES BEING USED:")
for i, col in enumerate(feature_columns):
    print(f"  {i}: {col}")

# PREPARE FINAL DATASET
df_final = df.sample(frac=1, random_state=42).reset_index(drop=True)
Y = df_final["nights"].to_numpy()
X = df_final[feature_columns].to_numpy()

def predict(w, x, b):
    return np.dot(w, x) + b

def compute_cost(X, Y, w, b):
    m = len(X)
    total_cost = 0
    
    for i in range(m):
        prediction = np.dot(w, X[i]) + b
        error = prediction - Y[i]
        total_cost += error ** 2
    
    cost = total_cost / (2 * m)
    return cost

def compute_gradient(X, Y, w, b):
    m = len(X)
    dw = np.zeros_like(w)
    db = 0
    
    for i in range(m):
        error = np.dot(w, X[i]) + b - Y[i]
        dw += error * X[i]
        db += error
    
    return dw / m, db / m

def gradient_descent(X, Y, w, b, learning_rate, iterations):
    cost_history = []
    
    for i in range(iterations):
        # Compute gradients
        dw, db = compute_gradient(X, Y, w, b)
        
        # Update parameters
        w -= learning_rate * dw
        b -= learning_rate * db
        
        # Compute and store cost every 100 iterations
        if i % 100 == 0:
            cost = compute_cost(X, Y, w, b)
            cost_history.append(cost)
            print(f"Iteration {i:4d}: Cost = {cost:.6f}")
    
    return w, b, cost_history

# SPLIT DATA
split_index = int(len(df_final) * 0.8)
X_train = X[:split_index]
y_train = Y[:split_index]
X_test = X[split_index:]
y_test = Y[split_index:]

print(f"\n📊 Training data: {X_train.shape[0]} samples")
print(f"📊 Test data: {X_test.shape[0]} samples")
print(f"📊 Number of features: {X_train.shape[1]}\n")

# INITIALIZE PARAMETERS
w = np.zeros(X_train.shape[1])
b = 0

# Calculate initial cost
initial_cost = compute_cost(X_train, y_train, w, b)
print(f"Initial Cost (before training): {initial_cost:.6f}\n")

# TRAIN MODEL
print("🔄 Training in progress...")
w, b, cost_history = gradient_descent(
    X_train, 
    y_train, 
    w, 
    b, 
    learning_rate=0.01, 
    iterations=2000
)

# FINAL EVALUATION
print(df.info())
final_cost = compute_cost(X_train, y_train, w, b)
test_cost = compute_cost(X_test, y_test, w, b)
print(f"\n✅ Training complete!")
print(f"Final Training Cost: {final_cost:.6f}")
print(f"Test Cost: {test_cost:.6f}")
print(f"Weights: {w}")
print(f"Bias: {b:.6f}")

# ACCURACY METRICS
predictions = np.array([predict(w, X_test[i], b) * nights_std + nights_mean for i in range(len(X_test))])
actuals = y_test * nights_std + nights_mean

mae = 0
for i in range(len(actuals)):
    mae += abs(predictions[i] - actuals[i])
mae = mae / len(actuals)

rmse = 0
for i in range(len(actuals)):
    rmse += (predictions[i] - actuals[i]) ** 2
rmse = (rmse / len(actuals)) ** 0.5

ss_res = 0
ss_tot = 0
actual_mean = sum(actuals) / len(actuals)
for i in range(len(actuals)):
    ss_res += (actuals[i] - predictions[i]) ** 2
    ss_tot += (actuals[i] - actual_mean) ** 2
r2 = 1 - (ss_res / ss_tot)

print(f"MAE:  {mae:.2f} nights")
print(f"RMSE: {rmse:.2f} nights")
print(f"R²:   {r2:.4f}")

🎯 FEATURES BEING USED:
  0: los_angeles
  1: new_york
  2: san_francisco
  3: Monday
  4: Tuesday
  5: Wednesday
  6: Thursday
  7: Friday
  8: Saturday
  9: Sunday
  10: price_per_night
  11: total_price

📊 Training data: 16000 samples
📊 Test data: 4000 samples
📊 Number of features: 12

Initial Cost (before training): 0.496918

🔄 Training in progress...
Iteration    0: Cost = 0.491122
Iteration  100: Cost = 0.227367
Iteration  200: Cost = 0.169822
Iteration  300: Cost = 0.153488
Iteration  400: Cost = 0.148541
Iteration  500: Cost = 0.147023
Iteration  600: Cost = 0.146555
Iteration  700: Cost = 0.146410
Iteration  800: Cost = 0.146364
Iteration  900: Cost = 0.146350
Iteration 1000: Cost = 0.146345
Iteration 1100: Cost = 0.146343
Iteration 1200: Cost = 0.146343
Iteration 1300: Cost = 0.146342
Iteration 1400: Cost = 0.146342
Iteration 1500: Cost = 0.146342
Iteration 1600: Cost = 0.146342
Iteration 1700: Cost = 0.146342
Iteration 1800: Cost = 0.146341
Iteration 1900: Cost = 0.146341
<cl

In [ ]:
df